# Step 01: Exploratory Data Analysis (EDA) of Annotations
### Statistical Profiling of Discovered Categories, Dimensions, Aspect Ratios & Tag Densities
Directly extracted from `multi_class_train_rfdetr.ipynb` (Cells 1, 4, 5, 6, 13).

### Key Analyses:
- Discovers annotation JSON files from `coco_files/`.
- Parses and prints category distribution with percentages and bar chart.
- Computes bounding box dimensions (width, height, area, aspect ratio).
- Analyzes tag density per image (single-tag vs multi-tag shelf scenes).
- Displays sample annotations per class in-memory (using Pure PIL & IPython - zero matplotlib disk leaks).

In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
except ImportError:

try:
except ImportError:

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
else:
logger.info("=" * 65)


In [ ]:
# CELL 2 - Path Configuration
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
INPUT_JSON_DIR = REPO_ROOT / "coco_files"
IMAGES_DIR = INPUT_JSON_DIR / "images"
logger.info(f"Input JSON Directory: {INPUT_JSON_DIR.resolve()}")
logger.info(f"Images Directory:     {IMAGES_DIR.resolve()}")

In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")

In [ ]:
# CELL 5 - Category Distribution Bar Chart
if total_boxes > 0:
    plt.figure(figsize=(10, 4))
    cat_names = [c["name"] for c in categories_list]
    cat_counts = [len(annotations_by_cat[c["name"]]) for c in categories_list]
    bars = plt.bar(cat_names, cat_counts, color=["#1976D2", "#388E3C", "#F57C00", "#7B1FA2", "#D32F2F"][:len(cat_names)])
    plt.title("Annotation Count per Discovered Category", fontsize=14, fontweight="bold")
    plt.xlabel("Category", fontsize=11)
    plt.ylabel("Box Count", fontsize=11)
    plt.xticks(rotation=15)
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + max(1, total_boxes * 0.01), f"{yval}", ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# CELL 6 - Bounding Box Scale & Aspect Ratio Distributions
if total_boxes > 0:
    widths, heights, aspect_ratios, areas = [], [], [], []
    for cat_name, anns in annotations_by_cat.items():
        for a in anns:
            bbox = a.get("bbox", [])
            if len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0:
                w, h = bbox[2], bbox[3]
                widths.append(w)
                heights.append(h)
                aspect_ratios.append(w / max(1.0, h))
                areas.append(w * h)
                
    fig, axs = plt.subplots(1, 3, figsize=(16, 4))
    axs[0].hist(widths, bins=30, color="#1976D2", edgecolor="black")
    axs[0].set_title("Box Width (px)")
    axs[0].set_xlabel("Width")
    
    axs[1].hist(heights, bins=30, color="#388E3C", edgecolor="black")
    axs[1].set_title("Box Height (px)")
    axs[1].set_xlabel("Height")
    
    axs[2].hist(aspect_ratios, bins=30, color="#F57C00", edgecolor="black")
    axs[2].set_title("Aspect Ratio (Width / Height)")
    axs[2].set_xlabel("Ratio")
    
    plt.tight_layout()
    plt.show()
    
    logger.info(f"Median Box Width:       {np.median(widths):.1f} px (Mean: {np.mean(widths):.1f} px)")
    logger.info(f"Median Box Height:      {np.median(heights):.1f} px (Mean: {np.mean(heights):.1f} px)")
    logger.info(f"Median Aspect Ratio:    {np.median(aspect_ratios):.2f}")

In [ ]:
# CELL 7 - Tag Density per Image Analysis
if annotations_by_image:
    tags_per_img = [len(anns) for anns in annotations_by_image.values()]
    logger.info("=" * 70)
    logger.info(" TAG DENSITY PROFILE PER IMAGE")
    logger.info("=" * 70)
    logger.info(f"   - Total unique images:           {len(tags_per_img)}")
    logger.info(f"   - Single-tag images (count=1):   {sum(1 for c in tags_per_img if c == 1)} ({sum(1 for c in tags_per_img if c == 1)/len(tags_per_img)*100:.1f}%)")
    logger.info(f"   - Multi-tag images  (count>1):   {sum(1 for c in tags_per_img if c > 1)} ({sum(1 for c in tags_per_img if c > 1)/len(tags_per_img)*100:.1f}%)")
    logger.info(f"   - Max tags in a single image:    {max(tags_per_img) if tags_per_img else 0}")
    logger.info(f"   - Mean tags per image:           {np.mean(tags_per_img):.2f}")
    logger.info("=" * 70)

In [ ]:
# CELL 13 - Print Sample Images per Tag (In-Memory Display - No Files Saved)
from PIL import Image, ImageDraw
from IPython.display import display
from collections import defaultdict

# High-contrast color palette for distinct multi-class boxes
SAMPLE_COLORS = [
    (230, 25, 75),   # Red
    (60, 180, 75),   # Green
    (255, 225, 25),  # Yellow
    (67, 99, 216),   # Blue
    (245, 130, 49),  # Orange
    (145, 30, 180),  # Purple
    (66, 212, 244),  # Cyan
    (240, 50, 230),  # Magenta
    (191, 239, 69),  # Lime
    (250, 190, 212), # Pink
    (70, 153, 144),  # Teal
    (220, 190, 255)  # Lavender
]

def print_sample_images_per_tag(annotation_path: str, image_dir: str, max_display_width: int = 700):
    """
    Selects sample images covering each discovered category tag, annotates
    bounding boxes with colored badges in memory, and prints/displays them
    inline directly in the notebook without saving any files to disk.
    """
    print("=" * 75)
    print("[CELL 13] Printing Sample Images with Category Tags (No Disk Saving)")
    print("=" * 75)

    with open(annotation_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    img_id_map = {im["id"]: im for im in data.get("images", [])}
    
    cat_to_images = defaultdict(list)
    img_to_anns = defaultdict(list)
    for ann in data.get("annotations", []):
        cid = ann.get("category_id")
        iid = ann.get("image_id")
        cat_to_images[cid].append(iid)
        img_to_anns[iid].append(ann)

    # Match with existing local image files
    existing_cat_images = {}
    for cid, iids in cat_to_images.items():
        valid_iids = [
            iid for iid in iids
            if iid in img_id_map and os.path.exists(os.path.join(image_dir, img_id_map[iid]["file_name"]))
        ]
        if valid_iids:
            existing_cat_images[cid] = valid_iids

    if not existing_cat_images:
        print("[WARN] No valid local images found to display.")
        return

    # Display an annotated sample for every distinct category tag
    for cid in sorted(existing_cat_images.keys()):
        cat_name = categories.get(cid, str(cid))
        sample_img_id = existing_cat_images[cid][0]
        meta = img_id_map[sample_img_id]
        fname = meta["file_name"]
        img_path = os.path.join(image_dir, fname)

        img = Image.open(img_path).convert("RGB")
        im_w, im_h = img.size
        draw = ImageDraw.Draw(img)
        anns = img_to_anns.get(sample_img_id, [])

        cat_counts = defaultdict(int)
        for ann in anns:
            raw_bbox = ann.get("bbox", [])
            if len(raw_bbox) != 4:
                continue
            x, y, w, h = [float(v) for v in raw_bbox]
            ann_cid = int(ann.get("category_id", 0))
            ann_cname = str(categories.get(ann_cid, ann_cid))
            cat_counts[ann_cname] += 1
            color = SAMPLE_COLORS[ann_cid % len(SAMPLE_COLORS)]

            # Clamp box to valid dimensions
            x0 = max(0.0, min(float(im_w - 1), x))
            y0 = max(0.0, min(float(im_h - 1), y))
            x1 = max(0.0, min(float(im_w - 1), x + w))
            y1 = max(0.0, min(float(im_h - 1), y + h))

            # Draw bounding box
            draw.rectangle([x0, y0, x1, y1], outline=color, width=4)

            # Draw colored label badge with tag name
            label_text = f" {ann_cname} "
            if hasattr(draw, "textbbox"):
                tb = draw.textbbox((x0, max(0.0, y0 - 18.0)), label_text)
                draw.rectangle(tb, fill=color)
                draw.text((x0, max(0.0, y0 - 18.0)), label_text, fill=(255, 255, 255))
            else:
                draw.text((x0 + 2, max(0.0, y0 - 14.0)), label_text, fill=color)

        summary_str = ", ".join(f"'{k}': {v}" for k, v in sorted(cat_counts.items()))
        print(f"\n>>> Target Tag: [{cat_name}] | File: {fname} ({im_w}x{im_h}) | Boxes: {len(anns)} ({summary_str})")

        # Scale in-memory copy for clean notebook view if wide
        if im_w > max_display_width:
            scale = max_display_width / im_w
            display_img = img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
        else:
            display_img = img

        # Directly print/display in notebook (Zero files saved to disk)
        display(display_img)

    print("\n" + "=" * 75)
    print("All category tag samples displayed inline successfully.")
    print("=" * 75)

# Aliases for compatibility with earlier calls
visualize_samples = lambda *args, **kwargs: print_sample_images_per_tag(TRAIN_ANN, str(IMAGES_DIR))
print_and_visualize_samples = visualize_samples

print_sample_images_per_tag(TRAIN_ANN, str(IMAGES_DIR))